## 0. Preparación del entorno

In [ ]:
# Importamos las librerías necesarias

import getpass
import json
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Literal

from pathlib import Path
from pypdf import PdfReader

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from jsonschema import Draft202012Validator
from openai import OpenAI

In [ ]:
# Configuramos la conexión con el LLM de OpenAI

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)


if not os.getenv("OPENAI_API_KEY"):
    print("La API Key no ha sido encontrada en el archivo .env.")
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce la OpenAI API key: ")

client = OpenAI()

#Configuramos un modelo de generación de respuestas y otro para hacer el embedding

GENERATION_MODEL = os.getenv("OPENAI_GENERATION_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

# Creamos un diccionario con los parámetros de configuración del LLM.

# El valor de temperature se mantiene bajo para basarse más estrictamente en la información de TENERIFE.pdf. Como estamos configurando explicitamente temperature entonces no es conveniente modificar top_p.

LLM_PARAMS = {
    "model": GENERATION_MODEL,
    "temperature": 0.2,       
    "max_tokens": 800      
}


print(f"Modelo de generación: {GENERATION_MODEL}")
print(f"Modelo de embeddings: {EMBEDDING_MODEL}")

Modelo de generación: gpt-4.1-mini
Modelo de embeddings: text-embedding-3-small


## 1. Preparación de la base documental

In [4]:


def cargar_documento_tenerife() -> list[dict[str, str]]:
    """Carga el documento TENERIFE.pdf desde la carpeta data."""
    
    file_path = Path("../data/TENERIFE.pdf")
    
    # Validación en caso de que no esté presente el pdf de Tenerife
    if not file_path.exists():
        raise FileNotFoundError(f"¡Error! No se encontró el archivo. Ruta buscada: {file_path.resolve()}")
        
    # Extracción del texto del PDF
    extracted_text = ""
    reader = PdfReader(file_path)
    
    for page in reader.pages:
        extracted_text += page.extract_text() + "\n"
        
    return [
        {
            "source": file_path.name,
            "text": extracted_text.strip()
        }
    ]

try:
    raw_docs = cargar_documento_tenerife()
    print(f"Documentos cargados: {len(raw_docs)}")
    print(f"- Archivo: {raw_docs[0]['source']} | Caracteres extraídos: {len(raw_docs[0]['text'])}")
except Exception as e:
    print(e)

Documentos cargados: 1
- Archivo: TENERIFE.pdf | Caracteres extraídos: 16202


## 2. Chunking